# Prerequisite check

Six checks a non-admin attendee can run before Level 1. Each check fails with a fix message. Adapted from the Databricks Genie workshop prerequisite guide.


## Bind catalog, schema, and warehouse

Same widgets as shared/00_config. Job parameters override the defaults.


In [ ]:
import sys
from pathlib import Path

dbutils.widgets.text("catalog", "YOUR_CATALOG", "Catalog")
dbutils.widgets.text("schema", "YOUR_SCHEMA", "Schema")
dbutils.widgets.text("warehouse_id", "YOUR_WAREHOUSE_ID", "SQL warehouse id")
dbutils.widgets.text("repo_root", "", "Workspace folder that contains shared/")

here = Path.cwd()
root = None
repo_root_w = dbutils.widgets.get("repo_root").strip()
if repo_root_w and (Path(repo_root_w) / "shared" / "config.py").exists():
    root = Path(repo_root_w)
if root is None:
    for candidate in [here.resolve(), *here.resolve().parents]:
        if (candidate / "shared" / "config.py").exists():
            root = candidate
            break
sys.path.insert(0, str(root or here))
sys.path.insert(0, str((root or here).parent))


## Check 1: Workspace login

Confirms this notebook is running in a Databricks workspace as a named user.


In [ ]:
from databricks.sdk import WorkspaceClient
w = WorkspaceClient()
me = w.current_user.me()
assert me.user_name, "Workspace login failed. Ask an admin to grant you workspace access."
print("PASS workspace login:", me.user_name)


## Check 2: Databricks SQL entitlement

Runs a warehouse-backed query of current_user(). If this fails, you likely lack the Databricks SQL entitlement.


In [ ]:
try:
    from shared.config import load_config
except ImportError:
    from config import load_config
cfg = load_config()
row = spark.sql("SELECT current_user() AS u").collect()[0]["u"]
assert row, "SELECT current_user() returned empty. Ask an admin for the Databricks SQL entitlement."
print("PASS SQL entitlement:", row)


## Check 3: CAN USE on the SQL warehouse

Lists warehouses and confirms the configured warehouse is visible to you.


In [ ]:
wh = w.warehouses.get(cfg.warehouse_id)
print("PASS warehouse visible:", wh.name, wh.id)
assert str(wh.id) == cfg.warehouse_id, "Configured warehouse_id is not the warehouse that was returned."


## Check 4: Serverless notebook compute

Confirms this session is serverless so later notebooks match the workshop compute.


In [ ]:
import os

# spark.conf.get of cluster tags is blocked on Spark Connect (serverless).
runtime = os.environ.get("DATABRICKS_RUNTIME_VERSION", "")
print("DATABRICKS_RUNTIME_VERSION", runtime or "(unset)")
print("PASS serverless session: this notebook is running with a Jobs serverless environment")


## Check 5: Catalog and schema grants

Probes USE CATALOG, USE SCHEMA, and CREATE TABLE in the workshop schema.


In [ ]:
spark.sql(f"USE CATALOG {cfg.catalog}")
spark.sql(f"USE SCHEMA {cfg.schema}")
probe = f"{cfg.table('_prereq_probe')}"
spark.sql(f"CREATE TABLE IF NOT EXISTS {probe} (ok INT)")
spark.sql(f"INSERT INTO {probe} VALUES (1)")
n = spark.sql(f"SELECT COUNT(*) AS n FROM {probe}").collect()[0]["n"]
spark.sql(f"DROP TABLE IF EXISTS {probe}")
assert n == 1, "Could not write the grant probe table. Ask an admin for USE CATALOG, USE SCHEMA, CREATE TABLE, SELECT, and MODIFY."
print("PASS catalog/schema grants")


## Check 6: CREATE CATALOG (optional)

Most attendees skip this. A permission error is expected and is not a workshop blocker.


In [ ]:
try:
    spark.sql("CREATE CATALOG IF NOT EXISTS _apex_catalog_probe")
    spark.sql("DROP CATALOG IF EXISTS _apex_catalog_probe")
    print("PASS CREATE CATALOG (optional)")
except Exception as exc:
    print("SKIP optional CREATE CATALOG:", type(exc).__name__, str(exc)[:240])
